# minigdb — Random Graph Generation with NetworkX

This notebook demonstrates generating a random graph with **NetworkX** (Erdős–Rényi model)
and loading it into **minigdb** for graph queries and visualization.

## What it covers
1. Generate a random graph with N nodes and edge-inclusion probability P
2. Assign random node attributes (department, seniority)
3. Assign random edge weights
4. **Bulk-insert via CSV** — uses `db.load_csv()` to load all nodes and edges in one call,
   bypassing the GQL parser for maximum throughput
5. Run GQL queries: degree distribution, shortest path, connected components
6. Compare minigdb query results against NetworkX ground truth
7. **Parameterized queries** — `$param` placeholders + `UNWIND $list AS item INSERT …`

## Requirements
```
pip install minigdb networkx
```

In [1]:
import csv
import io
import os
import random
import tempfile
import time

import minigdb
import networkx as nx

In [2]:
# Parameters
N    = 1000    # number of nodes
P    = 0.02    # probability that any given pair of nodes is connected
SEED = 42

random.seed(SEED)
print(f'Generating a graph with {N} nodes, {P:.2%} edge probability')
expectation_edges = P * N * (N - 1) / 2
print(f'Expected number of edges: ~{expectation_edges:.0f}')

Generating a graph with 1000 nodes, 2.00% edge probability
Expected number of edges: ~9990


In [3]:
db = minigdb.open('random_graph')

## 1. Generate the random graph with NetworkX

In [4]:
G = nx.erdos_renyi_graph(N, P, seed=SEED, directed=True)

DEPTS  = random.choices(['Engineering', 'Product', 'Sales', 'Design', 'Finance'], k=N)
LEVELS = random.choices(['Junior', 'Mid', 'Senior', 'Staff', 'Principal'],        k=N)
WEIGHTS = [round(random.random(), 3) for _ in range(G.number_of_edges())]

print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')

Nodes: 1000
Edges: 19994


## 2. Build CSV data in memory

We write the NetworkX graph to two in-memory CSV buffers using the
minigdb convention:

| Column | Role |
|--------|------|
| `:ID`  | user-assigned integer ID (used to link nodes and edges) |
| `:LABEL` | node label (all nodes get `Person`) |
| `:START_ID` / `:END_ID` | reference node `:ID` values |
| `:TYPE`  | edge label (`knows`) |
| other  | node / edge properties |

In [5]:
# ── Build node CSV ────────────────────────────────────────────────────────────
node_buf = io.StringIO()
node_writer = csv.writer(node_buf)
node_writer.writerow([':ID', ':LABEL', 'name', 'dept', 'level', 'node_id'])
for nid in G.nodes():
    node_writer.writerow([nid, 'Person', f'Name_{nid}', DEPTS[nid], LEVELS[nid], nid])

# ── Build edge CSV ────────────────────────────────────────────────────────────
edge_buf = io.StringIO()
edge_writer = csv.writer(edge_buf)
edge_writer.writerow([':START_ID', ':END_ID', ':TYPE', 'weight'])
for (i, j), w in zip(G.edges(), WEIGHTS):
    edge_writer.writerow([i, j, 'knows', w])

print(f'Node CSV rows : {G.number_of_nodes()}')
print(f'Edge CSV rows : {G.number_of_edges()}')
# Preview the first 3 rows of each
print('\nNode CSV (first 3 rows):')
print('\n'.join(node_buf.getvalue().splitlines()[:4]))
print('\nEdge CSV (first 3 rows):')
print('\n'.join(edge_buf.getvalue().splitlines()[:4]))

Node CSV rows : 1000
Edge CSV rows : 19994

Node CSV (first 3 rows):
:ID,:LABEL,name,dept,level,node_id
0,Person,Name_0,Design,Junior,0
1,Person,Name_1,Engineering,Staff,1
2,Person,Name_2,Product,Senior,2

Edge CSV (first 3 rows):
:START_ID,:END_ID,:TYPE,weight
0,20,knows,0.143
0,101,knows,0.66
0,125,knows,0.221


## 3. Bulk-insert into minigdb via CSV

`db.load_csv(nodes_path, edges_path)` loads both files atomically inside a
single RocksDB WriteBatch — **zero GQL parse overhead** per row.

In [6]:
# Clear any data from a previous run.
db.clear()

# Write CSV buffers to temporary files (load_csv takes file paths).
with tempfile.TemporaryDirectory() as tmpdir:
    nodes_path = os.path.join(tmpdir, 'nodes.csv')
    edges_path = os.path.join(tmpdir, 'edges.csv')

    with open(nodes_path, 'w', newline='') as f:
        f.write(node_buf.getvalue())
    with open(edges_path, 'w', newline='') as f:
        f.write(edge_buf.getvalue())

    t0 = time.perf_counter()
    result = db.load_csv(nodes_path, edges_path)
    elapsed_ms = (time.perf_counter() - t0) * 1000

print(f"Inserted {result['nodes_inserted']} nodes and {result['edges_inserted']} edges"
      f" in {elapsed_ms:.1f} ms")
if result['skipped']:
    print(f"  (skipped {result['skipped']} edges with unresolved IDs)")

# # Retrieve the id_map for later use (e.g. shortest path query).
# # The map is stored internally; re-load just the nodes CSV to get it back.
# with tempfile.TemporaryDirectory() as tmpdir:
#     nodes_path = os.path.join(tmpdir, 'nodes.csv')
#     with open(nodes_path, 'w', newline='') as f:
#         f.write(node_buf.getvalue())

# Build nx_id → ULID map from the database.
rows = db.query('MATCH (n:Person) RETURN n.node_id, n')
id_map = {int(r['n.node_id']): r['n'] for r in rows}   # nx_id → minigdb ULID
print(f'id_map built for {len(id_map)} nodes')

Inserted 1000 nodes and 19994 edges in 614.4 ms
id_map built for 1000 nodes


## 4. Verify counts

In [7]:
G.number_of_nodes()

1000

In [8]:
node_count = db.query('MATCH (n:Person) RETURN count(n) AS total')[0]['total']
edge_count = db.query('MATCH (a)-[r]->(b) RETURN count(r) AS total')[0]['total']

print(f'minigdb nodes : {node_count}  (expected {G.number_of_nodes()})')
print(f'minigdb edges : {edge_count}  (expected {G.number_of_edges()})')
assert node_count == G.number_of_nodes(), 'Node count mismatch!'
assert edge_count == G.number_of_edges(), 'Edge count mismatch!'
print('Counts match ✓')

minigdb nodes : 1000  (expected 1000)
minigdb edges : 19994  (expected 19994)
Counts match ✓


## 5. Degree distribution — top 10 highest out-degree nodes

In [9]:
rows = db.query(
    'MATCH (n:Person)-[r]->(m) '
    'RETURN n.name, n.dept, count(r) AS out_degree '
    'ORDER BY out_degree DESC '
    'LIMIT 10'
)
print(f'{"Name":<16} {"Dept":<14} Out-degree')
print('-' * 42)
for r in rows:
    print(f"{r['n.name']:<16} {r['n.dept']:<14} {r['out_degree']}")

Name             Dept           Out-degree
------------------------------------------
Name_986         Finance        37
Name_347         Product        36
Name_629         Design         34
Name_835         Sales          34
Name_323         Finance        33
Name_223         Design         32
Name_500         Design         32
Name_553         Design         32
Name_221         Sales          31
Name_235         Engineering    31


## 6. Nodes per department

In [10]:
rows = db.query(
    'MATCH (n:Person) '
    'RETURN n.dept, count(n) AS members '
    'ORDER BY members DESC'
)
print(f'{"Department":<16} Members')
print('-' * 28)
for r in rows:
    bar = '\u2588' * int(r['members'] // 20)  # scale bar to fit
    print(f"{r['n.dept']:<16} {r['members']:>3}  {bar}")

Department       Members
----------------------------
Finance          212  ██████████
Design           208  ██████████
Sales            206  ██████████
Product          196  █████████
Engineering      178  ████████


## 7. Shortest path between two specific nodes

We pick the pair (0, N-1) — first and last node — and compare minigdb's
`shortestPath` algorithm against NetworkX's `shortest_path`.

In [11]:
src_nx, dst_nx = 0, N - 1
src_ulid = id_map[src_nx]
dst_ulid = id_map[dst_nx]

# ── minigdb shortest path ─────────────────────────────────────────────────────
rows = db.query(
    f"CALL shortestPath(source: '{src_ulid}', target: '{dst_ulid}') "
    f"YIELD path, cost"
)
if rows:
    print(f'minigdb: shortest path length = {rows[0]["cost"]}')
else:
    print('minigdb: no path found')

# ── NetworkX ground truth ────────────────────────────────────────────────────
try:
    nx_path = nx.shortest_path(G, src_nx, dst_nx)
    print(f'NetworkX: shortest path length = {len(nx_path) - 1}')
    print(f'NetworkX path: {" \u2192 ".join(str(n) for n in nx_path)}')
except nx.NetworkXNoPath:
    print('NetworkX: no path found')

minigdb: shortest path length = 3.0
NetworkX: shortest path length = 3
NetworkX path: 0 → 101 → 452 → 999


## 8. Weakly-connected components via minigdb WCC

In [12]:
rows = db.query(
    'CALL wcc() YIELD nodeId, componentId '
    'RETURN componentId, count(nodeId) AS size '
    'ORDER BY size DESC '
    'LIMIT 5'
)
print('Top 5 weakly-connected components:')
print(f'{"Component":<30} Size')
print('-' * 38)
for r in rows:
    print(f"{str(r['componentId'])[:28]:<30} {r['size']}")

comp_rows = db.query(
    'CALL wcc() YIELD nodeId, componentId '
    'RETURN componentId, count(nodeId) AS sz '
    'ORDER BY sz DESC'
)
mgdb_comps = len(comp_rows)
nx_comps   = nx.number_weakly_connected_components(G)
print(f'\nTotal WCC count — minigdb: {mgdb_comps}, NetworkX: {nx_comps}')

Top 5 weakly-connected components:
Component                      Size
--------------------------------------
None                           0

Total WCC count — minigdb: 1, NetworkX: 1


## 9. PageRank — top 10 most influential nodes

In [13]:
rows = db.query(
    'CALL pageRank(iterations: 20, dampingFactor: 0.85) '
    'YIELD node, score '
    'RETURN node, score '
    'ORDER BY score DESC '
    'LIMIT 10'
)

ulid_to_name = {v: f'Person_{k}' for k, v in id_map.items()}

print(f'{"Node":<16} PageRank')
print('-' * 30)
for r in rows:
    node_id = r['node']
    name = ulid_to_name.get(node_id, (node_id[:12] + '…') if node_id else '(unknown)')
    print(f"{name:<16} {r['score']:.4f}")


Node             PageRank
------------------------------
Person_16        0.0017
Person_92        0.0016
Person_928       0.0016
Person_160       0.0015
Person_806       0.0015
Person_552       0.0015
Person_615       0.0015
Person_687       0.0015
Person_295       0.0015
Person_870       0.0015


## 10. Edge weight distribution — top 8 heaviest `knows` edges

In [14]:
rows = db.query(
    'MATCH (a)-[r:knows]->(b) '
    'RETURN a.name, b.name, r.weight '
    'ORDER BY r.weight DESC '
    'LIMIT 8'
)
if rows:
    print(f'{"From":<16} {"To":<16} Weight')
    print('-' * 42)
    for r in rows:
        print(f"{r['a.name']:<16} {r['b.name']:<16} {r['r.weight']:.3f}")
else:
    print('No knows edges found.')

From             To               Weight
------------------------------------------
Name_3           Name_215         1.000
Name_21          Name_432         1.000
Name_44          Name_951         1.000
Name_489         Name_632         1.000
Name_23          Name_375         0.999
Name_95          Name_451         0.999
Name_153         Name_354         0.999
Name_211         Name_662         0.999


## 11. Parameterized queries

minigdb supports `$param` placeholders in GQL.  Parameters are passed as a
Python dict via `db.query_with_params(gql, params)`.

### Why parameterized queries?
* **Safety** — no string interpolation, so no injection risk
* **Bulk inserts** — `UNWIND $list AS item INSERT …` loads a list in one round-trip
* **Reusability** — compile the GQL once, vary the params

In [ ]:
# Re-open the graph (it was closed at the end of section 10).
db = minigdb.open('random_graph')

# ── Scalar parameter: look up a person by name ────────────────────────────────
rows = db.query_with_params(
    'MATCH (n:Person) WHERE n.name = $name RETURN n.name, n.dept, n.level',
    {'name': 'Name_42'}
)
if rows:
    r = rows[0]
    print(f"Found: {r['n.name']}  dept={r['n.dept']}  level={r['n.level']}")
else:
    print('Not found')

# ── Numeric parameter: filter by out-degree threshold ─────────────────────────
rows = db.query_with_params(
    'MATCH (n:Person)-[r]->(m) '
    'RETURN n.name, count(r) AS out_degree '
    'ORDER BY out_degree DESC LIMIT $k',
    {'k': 5}
)
print(f'\nTop {len(rows)} nodes by out-degree (via $k=5):')
for r in rows:
    print(f"  {r['n.name']}  out-degree={r['out_degree']}")

In [ ]:
# ── UNWIND $list AS item INSERT — bulk-create project nodes ──────────────────
# This inserts a batch of "Project" nodes in a single GQL statement by passing
# the list as a parameter.  No loop, no string building — the database
# iterates the list internally.

projects = [
    {'id': 'P1', 'title': 'Graph Analytics',  'status': 'active'},
    {'id': 'P2', 'title': 'ML Pipeline',       'status': 'active'},
    {'id': 'P3', 'title': 'Data Warehouse',    'status': 'archived'},
    {'id': 'P4', 'title': 'Realtime Dashboard','status': 'active'},
    {'id': 'P5', 'title': 'Security Audit',    'status': 'active'},
]

result = db.query_with_params(
    'UNWIND $projects AS p INSERT (:Project {id: p.id, title: p.title, status: p.status})',
    {'projects': projects}
)

count = db.query('MATCH (p:Project) RETURN count(p) AS n')[0]['n']
print(f'Inserted {count} Project nodes via UNWIND $projects')

# Verify we can read them back.
rows = db.query("MATCH (p:Project) WHERE p.status = 'active' RETURN p.id, p.title ORDER BY p.id")
print(f'\nActive projects:')
for r in rows:
    print(f"  {r['p.id']}: {r['p.title']}")

In [ ]:
# ── UNWIND + MATCH+INSERT — assign people to projects by department ───────────
# Pass a list of {dept, project_id} mappings and create WORKS_ON edges.

assignments = [
    {'dept': 'Engineering', 'pid': 'P1'},
    {'dept': 'Design',      'pid': 'P4'},
    {'dept': 'Finance',     'pid': 'P2'},
    {'dept': 'Sales',       'pid': 'P3'},
    {'dept': 'Product',     'pid': 'P5'},
]

# For each assignment, connect matching people to the project.
# (Runs one GQL per assignment — demonstrates parameterized MATCH+INSERT.)
total_edges = 0
for a in assignments:
    db.query_with_params(
        'MATCH (person:Person {dept: $dept}), (proj:Project {id: $pid}) '
        'INSERT (person)-[:WORKS_ON]->(proj)',
        {'dept': a['dept'], 'pid': a['pid']}
    )
    rows = db.query_with_params(
        'MATCH (p:Person {dept: $dept})-[:WORKS_ON]->(proj:Project {id: $pid}) '
        'RETURN count(p) AS n',
        {'dept': a['dept'], 'pid': a['pid']}
    )
    n = rows[0]['n'] if rows else 0
    total_edges += n
    print(f"  {a['dept']:<14} → {a['pid']}  ({n} edges)")

print(f'\nTotal WORKS_ON edges created: {total_edges}')

## 12. Clean up

In [15]:
db.close()
print('Graph closed. Data persists at the minigdb data directory for future sessions.')
print('Re-open with: db = minigdb.open("random_graph")')

Graph closed. Data persists at the minigdb data directory for future sessions.
Re-open with: db = minigdb.open("random_graph")
